In [ ]:
# Imports
import pprint
from tokenize import Double

from jsonschema.validators import Draft7Validator
import json

from sqlalchemy.testing.schema import mapped_column

from src.mosdex.mosdex_db import MosdexBase

In [7]:
"""
Open a Mosdex file and validate it against the schema
"""

MOSDEX_SCHEMA_FILE = "../data/MOSDEXSchemaV2-1.json"
MOSDEX_FILE = "../data/sailco_2-1.json"

with open(MOSDEX_SCHEMA_FILE, "r") as f:
    schema = json.load(f)
validator = Draft7Validator(schema)

with open(MOSDEX_FILE, "r") as f:
    mosdex = json.load(f)

if not validator.is_valid(mosdex):
    print(f"File {MOSDEX_FILE} is not a valid Mosdex file.")
    pp = pprint.PrettyPrinter(indent=4)
    for error in sorted(validator.iter_errors(mosdex), key=str):
        print()
        pp.pprint(error.message)
else:
    print(f"File {MOSDEX_FILE} is a valid instance of schema {MOSDEX_SCHEMA_FILE}.")


File ../data/sailco_2-1.json is a valid instance of schema ../data/MOSDEXSchemaV2-1.json.


In [18]:
"""
Print some features of the MOSDEX_FILE
"""

print(f"The syntax of {MOSDEX_FILE} is {mosdex['SYNTAX']}")
for module in mosdex['MODULES']:
    print(f"\nModule {module['NAME']} is a {module['KIND']}")
    

The syntax of ../data/sailco_2-1.json is MOSDEX/MOSDEX v2-0/MOSDEXSchemaV2-0.json

Module sailco is a MODEL


In [24]:
# Get reference to a module that has KIND == MODEL
model = {}
for module in mosdex['MODULES']:
    if module['KIND'] == 'MODEL':
        model = module
        break

print(f"Got handle to MODEL: {model['NAME']}")
print(f"The sections of the model are {list(model.keys())}")
print(f"\tNAME: {model['NAME']}")
print(f"\tCLASS: {model['CLASS']}")
print(f"\tKIND: {model['KIND']}")
print(f"\tTABLES: there are {len(model['TABLES'])} tables:")
for table in model['TABLES']:
    print(f"\t\t{table['NAME']:10s} \t class/kind: {table['CLASS']}/{table['KIND']}")

Got handle to MODEL: sailco
The sections of the model are ['NAME', 'CLASS', 'KIND', 'HEADING', 'TABLES']
	NAME: sailco
	CLASS: MODULE
	KIND: MODEL
	TABLES: there are 11 tables:
		demands    	 class/kind: DATA/INPUT
		parameters 	 class/kind: DATA/INPUT
		regular    	 class/kind: VARIABLE/CONTINUOUS
		extra      	 class/kind: VARIABLE/CONTINUOUS
		inventory  	 class/kind: VARIABLE/CONTINUOUS
		cost       	 class/kind: VARIABLE/CONTINUOUS
		totalCost  	 class/kind: VARIABLE/CONTINUOUS
		ctCapacity 	 class/kind: CONSTRAINT/LINEAR
		ctBoat     	 class/kind: CONSTRAINT/LINEAR
		ctCost     	 class/kind: CONSTRAINT/LINEAR
		production 	 class/kind: DATA/OUTPUT


In [47]:
"""
Initialize the database engine
"""
from sqlalchemy import create_engine, Integer, String, Double

engine = create_engine('sqlite:///:memory:')

from sqlalchemy.orm import declarative_base, mapped_column

Base = declarative_base()

for table_ in model['TABLES']:
    table_name = table_['NAME']
    table_class = table_['CLASS']
    table_kind = table_['KIND']
    table_schema = table_['SCHEMA']
    
    db_table_class_name = table_['CLASS'] + '_' + model['NAME'] + '_' + table_['NAME']
    
    db_table_attr = { '__tablename__': table_name,
                      'id': mapped_column(Integer, primary_key=True)}
    
    db_table_col = {} 
    for name, kind in zip(table_schema['NAME'], table_schema['KIND']):
        if kind == 'INTEGER':
            type_col = Integer
        elif kind == 'DOUBLE' or kind == 'DOUBLE_FUNCTION':
            type_col = Double
        elif kind == 'STRING':
            type_col = String
        else:
            print(f"Error, type {kind} is not supported")
            break
        db_table_attr[name] = mapped_column(type_col)
        
    if 'KEYS' in table_schema:
        for key in table_schema['KEYS']:
            
        
    db_table_instance = type(db_table_class_name, (Base,), db_table_attr)
    
Base.metadata.create_all(engine)
        
print(f"Database tables created")
for table in Base.metadata.tables.keys():
    print(f"\t{table}")
    print(f"\t\t{Base.metadata.tables[table].columns.keys()}")
    

Database tables created
	demands
		['id', 'period', 'ancestor', 'demand']
	parameters
		['id', 'regularCost', 'extraCost', 'capacity', 'initialInventory', 'inventoryCost']
	regular
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	extra
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	inventory
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	cost
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	totalCost
		['id', 'Name', 'period', 'Column', 'LowerBound', 'UpperBound', 'value']
	ctCapacity
		['id', 'Name', 'period', 'Row', 'Sense', 'RHS', 'dual']
	ctBoat
		['id', 'Name', 'period', 'Row', 'Sense', 'RHS', 'dual']
	ctCost
		['id', 'Name', 'period', 'Row', 'Constant', 'Sense', 'Value']
	production
		['id', 'period', 'regular', 'extra', 'inventory', 'marginalCapacityValue']
